# Chroma · labelling and cleaning on Colab's GPU

Team 8 · AI Methods hackathon. This notebook runs the whole `data_prep/` pipeline on Google's GPU, then shows
the **visual library**, so you can see what each step does to each photo.

| Step | Script | What it adds |
|---|---|---|
| 1 | `label_time_season.py` | CLIP guesses place, time of day and season (GPU) |
| 2 | `filter_closeups.py` | removes close-ups: a person, animal or object as the subject |
| 3 | `filter_quality.py` | **new**: removes black-and-white, sepia, dark, blown-out, flat, blurry, framed and duplicate photos |
| 4 | `extract_palette.py` | colours and the music parameters |
| 5 | `extract_gradient.py` | one 8-stop gradient map per photo |
| 6 | `extract_gradient_layers.py` | **new**: Depth Anything splits the photo into near / mid / far, one gradient each (GPU) |
| 7 | `make_labels_all.py` | one table: `labels_all.csv` |

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells from the top.
Every step skips itself if its output already exists, so you can rerun safely after uploading a bigger zip.

In [ ]:
# 0 · GPU and the repository
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU"
%cd /content
!test -d hackathon-team8-chroma || git clone -q https://github.com/iaac-maai/hackathon-team8-chroma.git
%cd /content/hackathon-team8-chroma/01_autoencoder
!pip -q install scikit-image

## 1 · The new and updated scripts

These are not on GitHub yet, so the notebook writes them into `data_prep/`. Once they are pushed, you can skip this section.

In [ ]:
%%writefile data_prep/filter_quality.py
"""Second cleaning pass: remove photos that would confuse the colour music.

The close-up filter (filter_closeups.py) removes photos where a person or an object
is the subject. This pass removes photos that are bad as PICTURES of a place.
Every check is measured from the pixels (no AI), so it is fast and repeatable:

    black and white   almost no colour: old photos and greyscale scans
                      (colourfulness < 8, Hasler & Suesstrunk 2003)
    sepia / toned     colour, but only one warm hue: old toned prints
                      (colourfulness < 30, 90 % of pixels within 25 deg of one hue,
                      and that hue is brown-orange-yellow, 15-65 deg)
    too dark          mean lightness < 0.12: nothing to see, a black chord
    blown out         mean lightness > 0.92, or > 40 % of pixels pure white
    flat              lightness spread < 0.06: fog, a wall, a blank sky
    blurry            variance of the Laplacian < 40 (on a 256 px grey copy)
    frame / border    a solid band >= 6 % on two opposite sides (scans, letterbox)
    duplicate         a near copy of a photo already kept
                      (difference hash, Hamming distance <= 5 of 64 bits)

Input:  ../data/places_index.csv (+ own / wikimedia, see sources.py)
Output: ../data/quality.csv     file, keep_quality, reason_quality and the measures
        ../data/quality_removed.html   every removed photo with its reason, to check

Run from the 01_autoencoder/ folder, after filter_closeups.py:
    python data_prep/filter_quality.py
    python data_prep/filter_quality.py --report-only   (measure, but keep everything)
"""

import argparse
import html
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFilter

from sources import load_index

ROOT = Path(__file__).resolve().parent.parent
DATA = ROOT / "data"

BW_COLOURFUL = 8.0                 # below: black and white
SEPIA_COLOURFUL, SEPIA_HUE_SHARE, SEPIA_HUE_DEG = 30.0, 0.90, 25
SEPIA_HUES = (15, 65)              # brown to yellow, in degrees
DARK, BRIGHT, WHITE_SHARE = 0.12, 0.92, 0.40
FLAT = 0.06
BLUR = 40.0
BORDER = 0.06                      # share of width / height
DUP_BITS = 5


def colourfulness(rgb):
    """Hasler & Suesstrunk (2003). ~0 grey, ~15 dull, ~35 average, 60+ vivid."""
    r, g, b = (rgb[..., i].astype(np.float32) for i in range(3))
    rg, yb = r - g, 0.5 * (r + g) - b
    return float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))


def hue_share(img):
    """Main hue (degrees) and the share of coloured pixels within SEPIA_HUE_DEG of it."""
    hsv = np.asarray(img.convert("HSV"), dtype=np.float32)
    h, s = hsv[..., 0] * 360 / 255, hsv[..., 1] / 255
    h = h[s > 0.08]
    if len(h) < 50:
        return 0.0, 1.0
    hist, _ = np.histogram(h, bins=36, range=(0, 360))
    centre = (hist.argmax() + 0.5) * 10
    d = np.abs((h - centre + 180) % 360 - 180)
    return float(centre), float((d <= SEPIA_HUE_DEG).mean())


def sharpness(grey):
    lap = np.asarray(grey.filter(ImageFilter.Kernel((3, 3), [0, 1, 0, 1, -4, 1, 0, 1, 0], 1, 128)),
                     dtype=np.float32)
    return float(lap.var())


def border(light):
    """Widest solid band (share of the side) on two opposite sides."""
    def band(lines):                   # lines: rows from the edge inwards
        ref = lines[0].mean()
        n = 0
        for line in lines:
            if line.std() > 0.03 or abs(line.mean() - ref) > 0.05:
                break
            n += 1
        return n / len(lines) * 2      # we look at half of the image
    h, w = light.shape
    top, bottom = band(light[: h // 2]), band(light[::-1][: h // 2])
    left, right = band(light.T[: w // 2]), band(light.T[::-1][: w // 2])
    return float(max(min(top, bottom), min(left, right)))


def dhash(grey):
    a = np.asarray(grey.resize((9, 8), Image.Resampling.BILINEAR), dtype=np.int16)
    return np.packbits((a[:, 1:] > a[:, :-1]).ravel())


def measure(path):
    img = Image.open(path).convert("RGB")
    img.thumbnail((256, 256))
    rgb = np.asarray(img)
    grey = img.convert("L")
    light = np.asarray(grey, dtype=np.float32) / 255
    hue, share = hue_share(img)
    return {
        "colourful": round(colourfulness(rgb), 2),
        "main_hue": round(hue),
        "hue_share": round(share, 3),
        "lightness": round(float(light.mean()), 3),
        "white_share": round(float((light > 0.97).mean()), 3),
        "spread": round(float(light.std()), 3),
        "sharpness": round(sharpness(grey), 1),
        "border": round(border(light), 3),
    }, dhash(grey)


def verdict(m):
    if m["lightness"] < DARK:
        return "too dark"
    if m["lightness"] > BRIGHT or m["white_share"] > WHITE_SHARE:
        return "blown out"
    if m["colourful"] < BW_COLOURFUL:
        return "black and white"
    if (m["colourful"] < SEPIA_COLOURFUL and m["hue_share"] >= SEPIA_HUE_SHARE
            and SEPIA_HUES[0] <= m["main_hue"] <= SEPIA_HUES[1]):
        return "sepia / toned"
    if m["spread"] < FLAT:
        return "flat"
    if m["sharpness"] < BLUR:
        return "blurry"
    if m["border"] >= BORDER:
        return "frame / border"
    return ""


def report(df):
    out = df[df["keep_quality"] == 0]
    tiles = "\n".join(
        f'<figure><img loading="lazy" src="{html.escape(r.file)}"><figcaption><b>{html.escape(r.reason_quality)}</b>'
        f'<br>{html.escape(str(r.place))} · c {r.colourful} · L {r.lightness} · sharp {r.sharpness}</figcaption></figure>'
        for r in out.sort_values(["reason_quality", "place"]).itertuples())
    (DATA / "quality_removed.html").write_text(f"""<!doctype html><meta charset="utf-8">
<title>Removed by quality filter</title>
<style>body{{font:13px system-ui;margin:16px;background:#f6f5f2}}
div{{display:grid;grid-template-columns:repeat(auto-fill,minmax(170px,1fr));gap:10px}}
figure{{margin:0;background:#fff;border:1px solid #ddd;border-radius:8px;overflow:hidden}}
img{{width:100%;aspect-ratio:1;object-fit:cover;display:block}}figcaption{{padding:6px}}</style>
<h1>Removed by the quality filter: {len(out)} photos</h1><div>{tiles}</div>""", encoding="utf-8")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--report-only", action="store_true", help="measure but keep every photo")
    args = ap.parse_args()

    index = load_index()
    closeups = DATA / "closeups.csv"
    kept_before = (set(pd.read_csv(closeups).query("keep == 1")["file"]) if closeups.exists()
                   else set(index["file"]))

    rows, hashes = [], []           # hashes of photos kept so far
    for i, f in enumerate(index["file"], 1):
        try:
            m, h = measure(DATA / f)
        except Exception as e:      # unreadable file
            rows.append({"file": f, "keep_quality": 0, "reason_quality": f"unreadable ({type(e).__name__})"})
            continue
        reason = verdict(m)
        if not reason and f in kept_before:
            if any(np.unpackbits(h ^ k).sum() <= DUP_BITS for k in hashes):
                reason = "duplicate"
            else:
                hashes.append(h)
        rows.append({"file": f, "keep_quality": int(args.report_only or not reason),
                     "reason_quality": reason, **m})
        if i % 500 == 0:
            print(f"{i}/{len(index)}")

    df = index[["file", "place"]].merge(pd.DataFrame(rows), on="file")
    if args.report_only:            # show what WOULD be removed
        df["keep_quality"] = (df["reason_quality"] == "").astype(int)
    report(df)
    df.drop(columns="place").assign(keep_quality=1 if args.report_only else df["keep_quality"]) \
      .to_csv(DATA / "quality.csv", index=False)

    both = df[df["file"].isin(kept_before)]
    print(f"\nOf the {len(both)} photos the close-up filter kept, this pass removes "
          f"{int((both['keep_quality'] == 0).sum())}:")
    print(both[both["keep_quality"] == 0].groupby(["place", "reason_quality"]).size().unstack(fill_value=0))
    print("\nKept per place after both filters:")
    print(both[both["keep_quality"] == 1]["place"].value_counts())
    print("\nCheck data/quality_removed.html before trusting the thresholds.")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile data_prep/extract_gradient_layers.py
"""Gradient maps in three depth layers: near, middle and far.

extract_gradient.py gives one gradient for the whole photo. Here the photo is
first split by DEPTH, so the foreground and the background each get their own
gradient: a warm market stall in front and a cool evening sky behind become
two different gradients instead of one average.

    1. Depth Anything V2 (Small, ready-made, no training) estimates how far away
       every pixel is (relative depth: it knows near from far, not metres);
    2. the pixels are cut into three layers with the same number of pixels:
       near = closest third, mid = middle third, far = farthest third
       (equal thirds, so every photo always has all three layers: a metro
       platform without sky still gets a "far");
    3. each layer gets its own 8-stop gradient map, exactly as in
       extract_gradient.py (Lab, sorted dark -> light, 8 equal groups).

Every photo becomes 3 layers x 8 stops x (L, a, b) = 72 numbers, always in the
order near, mid, far and dark -> light.

Output:
    ../data/gradient_layers.csv           file, near/mid/far stops (hex + lightness), 72 numbers
    ../data/gradient_layers/<name>.png    photo | depth | layers | repainted, with the 3 gradient bars

Run from the 01_autoencoder/ folder (after extract_gradient.py):
    pip install transformers torch
    python data_prep/extract_gradient_layers.py
    python data_prep/extract_gradient_layers.py --limit 20     (quick test on 20 photos)
"""

import argparse
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from skimage.color import lab2rgb, rgb2lab
from transformers import AutoImageProcessor, AutoModelForDepthEstimation

from extract_gradient import SIZE, STOPS, gradient_of, to_hex
from sources import load_index

ROOT = Path(__file__).resolve().parent.parent
DATA = ROOT / "data"
OUT = DATA / "gradient_layers"
DEPTH_MODEL = "depth-anything/Depth-Anything-V2-Small-hf"
LAYERS = ["near", "mid", "far"]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TINT = {"near": (230, 120, 60), "mid": (90, 170, 90), "far": (80, 130, 220)}   # for the picture only


def depth_of(img, proc, model):
    """Relative depth at SIZE x SIZE. Depth Anything returns inverse depth: larger = closer."""
    with torch.inference_mode():
        pred = model(**proc(images=img, return_tensors="pt").to(DEVICE)).predicted_depth.float().cpu()      # (1, h, w)
    pred = torch.nn.functional.interpolate(pred[None], size=(SIZE, SIZE), mode="bicubic",
                                           align_corners=False)[0, 0]
    return pred.numpy()


def layers_of(depth):
    """0 = near, 1 = mid, 2 = far, in three equal thirds of the pixels."""
    order = np.argsort(-depth.ravel())               # closest first
    layer = np.empty(depth.size, dtype=np.int8)
    for k, part in enumerate(np.array_split(order, 3)):
        layer[part] = k
    return layer.reshape(depth.shape)


def bar(stops, w):
    xs = np.linspace(0, STOPS - 1, w)
    lab = np.stack([np.interp(xs, np.arange(STOPS), stops[:, k]) for k in range(3)], axis=1)
    return np.repeat(np.clip(lab2rgb(lab[None]), 0, 1), 12, axis=0)


def picture(rgb, depth, layer, all_stops):
    """Four panels side by side, and the near / mid / far gradient bars underneath."""
    d = (depth - depth.min()) / (np.ptp(depth) + 1e-6)
    tint = np.array([TINT[n] for n in LAYERS], dtype=np.float64)[layer] / 255
    lab = rgb2lab(rgb)
    mapped = np.zeros_like(rgb)
    for k, stops in enumerate(all_stops):            # repaint each layer through its own gradient
        m = layer == k
        L = lab[..., 0][m]
        mapped[m] = np.clip(lab2rgb(np.stack([np.interp(L, stops[:, 0], stops[:, j]) for j in range(3)],
                                             axis=1)[None])[0], 0, 1)
    top = np.hstack([rgb, np.repeat(d[..., None], 3, axis=2), 0.5 * rgb + 0.5 * tint, mapped])
    w = top.shape[1]
    bars = np.vstack([bar(s, w) for s in all_stops])
    return Image.fromarray((np.vstack([top, bars]) * 255).astype(np.uint8))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--limit", type=int, default=0, help="only the first N photos (to test)")
    args = ap.parse_args()

    index = load_index()
    if args.limit:
        index = index.head(args.limit)
    OUT.mkdir(exist_ok=True)

    proc = AutoImageProcessor.from_pretrained(DEPTH_MODEL)
    model = AutoModelForDepthEstimation.from_pretrained(DEPTH_MODEL).eval().to(DEVICE)
    print("device:", DEVICE)

    rows = []
    for i, f in enumerate(index["file"], 1):
        img = Image.open(DATA / f).convert("RGB")
        depth = depth_of(img, proc, model)
        rgb = np.asarray(img.resize((SIZE, SIZE), Image.Resampling.BILINEAR), dtype=np.float64) / 255
        layer = layers_of(depth)

        all_stops = [gradient_of(rgb[layer == k][None])[0] for k in range(3)]
        name = Path(f).with_suffix(".png").name
        picture(rgb, depth, layer, all_stops).save(OUT / name)

        row = {"file": f, "gradient_layers_png": f"gradient_layers/{name}"}
        for n, stops in zip(LAYERS, all_stops):
            row[f"{n}_stops"] = json.dumps([{"hex": to_hex(s), "L": round(float(s[0]) / 100, 3)} for s in stops])
        for n, stops in zip(LAYERS, all_stops):
            for k, s in enumerate(stops, 1):
                row[f"{n}_s{k}_L"], row[f"{n}_s{k}_a"], row[f"{n}_s{k}_b"] = (round(float(v), 2) for v in s)
        rows.append(row)
        if i % 100 == 0:
            print(f"{i}/{len(index)}", flush=True)

    pd.DataFrame(rows).to_csv(DATA / "gradient_layers.csv", index=False)
    print(f"Wrote {len(rows)} layered gradient maps to {OUT} and {DATA / 'gradient_layers.csv'}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile data_prep/label_time_season.py
"""Add layers to every photo: place (with confidence), time of day and season.

Places365 has no such labels, so a ready-made image-text model (CLIP) guesses
them "zero-shot": it compares each photo with short text descriptions and picks
the closest one. Every guess keeps its confidence (0-1), so the library can show
which labels are uncertain and need a human check.

Input:  every source in sources.py (Places365, own photos, Wikimedia)
Output: ../data/labels_auto.csv   index columns + time, time_conf, season, season_conf
                                  + one probability column per label

Run from the 01_autoencoder/ folder:
    python data_prep/label_time_season.py
"""

from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

from sources import load_index

ROOT = Path(__file__).resolve().parent.parent
DATA = ROOT / "data"
MODEL = "openai/clip-vit-base-patch32"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"     # Colab GPU when there is one

# Several descriptions per label; their scores are averaged (prompt ensembling).
# Place: our six places, plus "other" for everything else (museums, food, documents ...).
PLACE = {
    "market": ["a photo of a food market with stalls", "a busy market hall with fruit and vegetables"],
    "street": ["a photo of a city street with buildings", "a narrow old town street", "a wide avenue with traffic"],
    "beach": ["a photo of a beach and the sea", "a sandy beach with people and waves"],
    "mountains": ["a view over a city from a hill", "a photo of hills and mountains", "a forest path on a hill"],
    "metro": ["a photo of a metro station platform", "an underground train station"],
    "airport": ["a photo of an airport terminal", "an airport hall with check-in desks"],
    "other": ["a photo of a museum interior", "a photo of a painting or artwork", "a photo of food on a plate",
              "a scanned document or map", "a close-up of a building detail or sculpture", "a photo of a church interior"],
}
INDOOR_PLACES = {"metro", "airport"}

TIME = {
    "day": [
        "a photo taken in bright daylight",
        "a place at midday under a clear sky",
        "a daytime photo",
    ],
    "golden hour": [
        "a photo taken at sunset",
        "a photo taken at sunrise with warm orange light",
        "a place at dusk with a golden sky",
    ],
    "night": [
        "a photo taken at night",
        "a place at night lit by street lights",
        "a dark night-time photo",
    ],
}
SEASON = {
    "spring": [
        "a photo taken in spring with blossoming flowers",
        "fresh green leaves and flowers in spring",
    ],
    "summer": [
        "a photo taken in summer with hot sun",
        "people in t-shirts and shorts on a summer day",
    ],
    "autumn": [
        "a photo taken in autumn with orange and brown leaves",
        "fallen leaves in autumn",
    ],
    "winter": [
        "a photo taken in winter with snow",
        "bare trees and people in coats in winter",
    ],
}


# Is the PLACE the subject, or a person, animal or object in front of it?
# Only "scene" photos are kept: close-ups do not show what a place looks like.
SUBJECT = {
    "scene": [
        "a wide photo of a city street with buildings",
        "a view over a city and its neighbourhoods",
        "a wide view of a public square or market",
        "a wide view of a beach and the sea",
        "a landscape photo of hills and mountains",
        "the inside of a large station or terminal hall",
    ],
    "person": [
        "a portrait photo of a person",
        "a close-up photo of a person posing for the camera",
        "a photo of a child",
        "a selfie",
    ],
    "animal": [
        "a close-up photo of a dog",
        "a close-up photo of an animal",
    ],
    "object": [
        "a close-up photo of an object",
        "a product photo of a single thing",
        "a close-up photo of a chair",
    ],
}


def as_tensor(out):
    """transformers 5 returns an output object; older versions return the tensor."""
    return out if isinstance(out, torch.Tensor) else out.pooler_output


def text_features(model, processor, label_prompts):
    """One normalised text vector per label (mean of its prompts)."""
    feats = []
    for prompts in label_prompts.values():
        inputs = processor(text=prompts, return_tensors="pt", padding=True).to(DEVICE)
        with torch.no_grad():
            f = as_tensor(model.get_text_features(**inputs))
        f = f / f.norm(dim=-1, keepdim=True)
        f = f.mean(dim=0)
        feats.append(f / f.norm())
    return torch.stack(feats)


def main():
    index = load_index()
    print(index["source"].value_counts().to_string())
    model = CLIPModel.from_pretrained(MODEL).eval().to(DEVICE)
    print("device:", DEVICE)
    processor = CLIPProcessor.from_pretrained(MODEL)

    t_feats = text_features(model, processor, TIME)
    s_feats = text_features(model, processor, SEASON)
    c_feats = text_features(model, processor, SUBJECT)
    p_feats = text_features(model, processor, PLACE)
    scale = model.logit_scale.exp()

    rows = []
    batch = 128 if DEVICE == "cuda" else 32
    for start in range(0, len(index), batch):
        part = index.iloc[start:start + batch]
        images = [Image.open(DATA / f).convert("RGB") for f in part["file"]]
        if start % 1600 == 0:
            print(f"{start:,}/{len(index):,}", flush=True)
        inputs = processor(images=images, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            img = as_tensor(model.get_image_features(**inputs))
        img = img / img.norm(dim=-1, keepdim=True)
        p_time = (scale * img @ t_feats.T).softmax(dim=-1)
        p_season = (scale * img @ s_feats.T).softmax(dim=-1)
        p_subject = (scale * img @ c_feats.T).softmax(dim=-1)
        p_place = (scale * img @ p_feats.T).softmax(dim=-1)

        for (_, r), pt, ps, pc, pp in zip(part.iterrows(), p_time, p_season, p_subject, p_place):
            row = r.to_dict()
            pi = int(pp.argmax())
            row["place_clip"], row["place_conf"] = list(PLACE)[pi], round(float(pp[pi]), 3)
            if not row.get("place"):                      # Wikimedia: no place label of its own
                row["place"] = row["place_clip"]
                row["indoor"] = int(row["place"] in INDOOR_PLACES)
            ci = int(pc.argmax())
            row["subject"] = list(SUBJECT)[ci]
            row["subject_conf"] = round(float(pc[ci]), 3)
            row["keep"] = int(row["subject"] == "scene")
            ti, si = int(pt.argmax()), int(ps.argmax())
            row["time"] = list(TIME)[ti]
            row["time_conf"] = round(float(pt[ti]), 3)
            # Season is not visible indoors (metro, airport terminal, covered market).
            if r["indoor"]:
                row["season"], row["season_conf"] = "unknown", 0.0
            else:
                row["season"] = list(SEASON)[si]
                row["season_conf"] = round(float(ps[si]), 3)
            for k, p in zip(TIME, pt):
                row[f"p_time_{k.replace(' ', '_')}"] = round(float(p), 3)
            for k, p in zip(SEASON, ps):
                row[f"p_season_{k}"] = round(float(p), 3)
            rows.append(row)

    out = pd.DataFrame(rows)
    out.to_csv(DATA / "labels_auto.csv", index=False)
    print(f"\nWrote {DATA / 'labels_auto.csv'}")
    print(pd.crosstab(out["place"], out["subject"]))
    print(f"Kept {out['keep'].sum()} of {len(out)} photos as scenes.\n")
    print(pd.crosstab(out["place"], out["time"]))
    print(pd.crosstab(out["place"], out["season"]))

    # How often is CLIP right? Only our own photos have true labels.
    if "time_true" in out:
        mine = out[(out["source"] == "own") & (out["time_true"].fillna("") != "")]
        if len(mine):
            t_acc = (mine["time"] == mine["time_true"]).mean()
            s = mine[mine["season"] != "unknown"]
            s_acc = (s["season"] == s["season_true"]).mean() if len(s) else float("nan")
            print(f"\nCLIP vs EXIF on {len(mine)} own photos: "
                  f"time of day {t_acc:.0%} right, season {s_acc:.0%} right")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile data_prep/make_labels_all.py
"""Merge every label into one table: one row per photo, every source.

    place, place_conf        Places365 label, own folder, or CLIP's guess (Wikimedia)
    time, season             CLIP's guess with confidence; *_true from the capture date when known
    keep, reason             close-up filter, then the quality filter (black and white,
                             sepia, dark, blown out, flat, blurry, border, duplicate); Wikimedia photos that CLIP calls "other"
                             (museum, artwork, food, document ...) are not kept either
    colours                  brightness, saturation, contrast, warmth, n_colours, palette
    gradient map             8 stops dark -> light, and the repainted image
    gradient layers          8 stops each for the near, mid and far depth thirds
    music                    scale, chords, sevenths, tone, melody, drums

Output: ../data/labels_all.csv

Run from the 01_autoencoder/ folder, after the other scripts:
    python data_prep/make_labels_all.py
"""

import json
from pathlib import Path

import pandas as pd

DATA = Path(__file__).resolve().parent.parent / "data"


def main():
    lab = pd.read_csv(DATA / "labels_auto.csv", keep_default_na=False, low_memory=False)
    cl = pd.read_csv(DATA / "closeups.csv", keep_default_na=False)[["file", "keep", "reason"]]
    pal = pd.read_csv(DATA / "palette.csv")
    gm = pd.read_csv(DATA / "gradient_maps.csv")[["file", "stops", "gradient_png"]]
    df = (lab.drop(columns=["keep"], errors="ignore")
             .merge(cl, on="file", how="left").merge(pal, on="file", how="left").merge(gm, on="file", how="left")
             .rename(columns={"stops": "gradient_stops", "palette": "palette_colours"}))
    qual = DATA / "quality.csv"      # second cleaning pass (filter_quality.py)
    if qual.exists():
        q = pd.read_csv(qual, keep_default_na=False)[["file", "keep_quality", "reason_quality"]]
        df = df.merge(q, on="file", how="left")
        bad = (df["keep_quality"] == 0) & (df["keep"] == 1)
        df.loc[bad, "keep"] = 0
        df.loc[bad, "reason"] = df.loc[bad, "reason_quality"]
        df = df.drop(columns=["keep_quality", "reason_quality"])
    layers = DATA / "gradient_layers.csv"   # near / mid / far gradients (extract_gradient_layers.py)
    if layers.exists():
        df = df.merge(pd.read_csv(layers)[["file", "near_stops", "mid_stops", "far_stops", "gradient_layers_png"]],
                      on="file", how="left")
    det = DATA / "details.csv"
    if det.exists():
        df = df.merge(pd.read_csv(det), on="file", how="left")
    other = df["place"] == "other"
    df.loc[other, "keep"] = 0
    df.loc[other & (df["reason"] == ""), "reason"] = "not one of our places"

    cols = ["file", "source", "place", "place_clip", "place_conf", "places365_category", "indoor",
            "keep", "reason", "lat", "lon", "taken_at",
            "time", "time_conf", "time_true", "season", "season_conf", "season_true",
            "brightness", "saturation", "contrast", "warmth", "n_colours", "palette_colours",
            "gradient_stops", "gradient_png", "near_stops", "mid_stops", "far_stops", "gradient_layers_png",
            "scale", "chords", "sevenths", "tone", "melody", "drums",
            "top_elements", "elements", "fam_architecture", "fam_street_furniture", "fam_furniture",
            "fam_paving", "fam_nature", "fam_sky", "fam_people", "fam_vehicles", "fam_goods",
            "paving_material", "paving_material_conf", "architecture_style", "architecture_style_conf",
            "facade_detail", "facade_detail_conf", "clothing", "clothing_conf", "segmentation_png",
            "licence", "author", "url"]
    df = df[[c for c in cols if c in df.columns]]
    df.to_csv(DATA / "labels_all.csv", index=False)

    ok = lambda s: isinstance(s, str) and len(json.loads(s)) > 0  # noqa: E731
    print(f"{len(df):,} photos  ·  kept {int(df['keep'].sum()):,}")
    print(f"  place        {(df['place'] != '').sum():,}")
    print(f"  time of day  {df['time'].isin(['day', 'golden hour', 'night']).sum():,}")
    print(f"  palette      {df['palette_colours'].apply(ok).sum():,}")
    print(f"  gradient     {df['gradient_stops'].apply(ok).sum():,}")
    if "elements" in df:
        print(f"  details      {df['elements'].apply(ok).sum():,}")
    k = df[df["keep"] == 1]
    print(pd.crosstab([k["source"], k["place"]], k["time"], margins=True))


if __name__ == "__main__":
    main()

## 2 · Your data zip

Put the zip on Google Drive (fastest) and set `ZIP_PATH`, or leave it empty to upload from your laptop.
The cell finds the data inside the zip by itself: either a ready `data/` folder (with `places_index.csv`)
or the raw Places365 tars (`val_256.tar` + `filelist_places365-standard.tar`).

**Starting with a partial zip is fine.** When the full zip is ready, upload it, set `REDO = True` in section 3 and run again.

In [ ]:
import shutil, subprocess, zipfile
from pathlib import Path

ZIP_PATH = "/content/drive/MyDrive/chroma_data.zip"   # <- your zip on Drive; "" = upload from laptop

DATA = Path("data"); RAW = DATA / "raw"; RAW.mkdir(parents=True, exist_ok=True)

if ZIP_PATH.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
    zip_file = Path(ZIP_PATH)
else:
    from google.colab import files
    up = files.upload()                                 # pick the zip
    zip_file = Path(next(iter(up)))

print("unzipping", zip_file, round(zip_file.stat().st_size / 1e6), "MB")
X = Path("/content/unzipped"); shutil.rmtree(X, ignore_errors=True)
with zipfile.ZipFile(zip_file) as z:
    z.extractall(X)

index = next(X.rglob("places_index.csv"), None)
tars = {p.name: p for p in X.rglob("*.tar")}
if index:                                               # a ready data/ folder
    subprocess.run(["rsync", "-a", f"{index.parent}/", f"{DATA}/"], check=True)
    print("copied the data folder from", index.parent)
for name, p in tars.items():                            # raw Places365 tars
    shutil.move(str(p), RAW / name); print("raw:", name)
if not index and "val_256.tar" not in tars:
    print("Found neither places_index.csv nor val_256.tar. The zip holds:")
    for p in sorted(X.rglob("*"))[:40]: print(" ", p.relative_to(X))

## 3 · Run the pipeline

Each step runs only if its output is missing (or if `REDO = True`). The two GPU steps take a few minutes for
900 photos; on a laptop CPU they took much longer.

In [ ]:
import subprocess, sys, time
REDO = False

def step(script, output, *args):
    out = DATA / output
    if out.exists() and not REDO:
        print(f"skip  {script:30s} ({output} exists)"); return
    t = time.time(); print(f"run   {script} {' '.join(args)}", flush=True)
    r = subprocess.run([sys.executable, f"data_prep/{script}", *args], capture_output=True, text=True)
    print("\n".join(r.stdout.strip().splitlines()[-12:]))
    if r.returncode:
        print(r.stderr[-3000:]); raise SystemExit(f"{script} failed")
    print(f"done  {script} in {time.time() - t:.0f} s\n")

if not (DATA / "places_index.csv").exists():
    step("prepare_places365.py", "places_index.csv")      # from the raw tars
step("label_time_season.py", "labels_auto.csv")           # GPU
step("filter_closeups.py", "closeups.csv")
step("filter_quality.py", "quality.csv")
step("extract_palette.py", "palette.csv")
step("extract_gradient.py", "gradient_maps.csv")
step("extract_gradient_layers.py", "gradient_layers.csv") # GPU
REDO_LAST, REDO = REDO, True
step("make_labels_all.py", "labels_all.csv")              # always rebuilt: it only merges
REDO = REDO_LAST

## 4 · The visual library

Every card is one photo, top to bottom:

- **the photo**, with its labels: place, time of day and season, with the confidence CLIP gave each guess (dashed = unsure, < 0.5);
- **kept or removed**, and why (close-up filter or quality filter);
- **gradient**: the one 8-stop map of the whole photo, dark → light;
- **near / mid / far**: the three depth-layer gradients. Compare them with the single gradient to see what the layers add;
- **the depth panel**: photo · depth (white = close) · the three layers (orange = near, green = mid, blue = far) · the photo repainted layer by layer.

Use the menus to filter. Choose *removed* to check the cleaning: is every removal right?

In [ ]:
import base64, html, io, json
import pandas as pd
import ipywidgets as w
from IPython.display import HTML, display
from PIL import Image

lab = pd.read_csv(DATA / "labels_all.csv", keep_default_na=False, low_memory=False)

def b64(path, size=None):
    p = DATA / path
    if not path or not p.exists(): return ""
    im = Image.open(p).convert("RGB")
    if size: im.thumbnail(size)
    buf = io.BytesIO(); im.save(buf, "JPEG", quality=82)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

def bar(stops_json, label):
    try: stops = json.loads(stops_json)
    except Exception: return ""
    cells = "".join(f'<span style="background:{s["hex"]}"></span>' for s in stops)
    return f'<div class="g"><i>{label}</i><div class="bar">{cells}</div></div>'

def chip(text, conf=None):
    low = conf not in (None, "") and float(conf) < 0.5
    c = f" {float(conf):.2f}" if conf not in (None, "") else ""
    return f'<span class="chip{" low" if low else ""}">{html.escape(str(text))}{c}</span>'

CSS = """<style>
.lib{display:grid;grid-template-columns:repeat(auto-fill,minmax(250px,1fr));gap:14px;font:12px system-ui}
.card{border:1px solid #ddd;border-radius:10px;overflow:hidden;background:#fff;color:#222}
.card.out{opacity:.85;outline:2px solid #c2410c}
.card>img.ph{width:100%;aspect-ratio:1;object-fit:cover;display:block}
.m{padding:6px 8px;display:flex;flex-wrap:wrap;gap:4px}
.chip{background:#efede8;border-radius:99px;padding:1px 7px}
.chip.low{outline:1px dashed #c2410c;color:#c2410c}
.why{padding:0 8px 6px;font-weight:600;color:#c2410c}
.g{display:grid;grid-template-columns:34px 1fr;align-items:center;gap:6px;padding:2px 8px}
.g i{font-style:normal;color:#777;font-size:11px}
.bar{display:flex;height:12px;border-radius:3px;overflow:hidden}.bar span{flex:1}
.dp{width:100%;display:block;margin-top:6px;border-top:1px solid #eee}
</style>"""

def card(r):
    keep = str(r.get("keep", "1")) in ("1", "1.0")
    out = "" if keep else f'<div class="why">removed: {html.escape(str(r.get("reason", "")))}</div>'
    chips = chip(r["place"], r.get("place_conf")) + chip(r.get("time", ""), r.get("time_conf")) + \
            chip(r.get("season", ""), r.get("season_conf") or None)
    layers = "".join(bar(r.get(f"{n}_stops", ""), n) for n in ("near", "mid", "far"))
    dp = b64(r.get("gradient_layers_png", ""))
    return (f'<div class="card{"" if keep else " out"}"><img class="ph" src="{b64(r["file"], (256, 256))}">'
            f'<div class="m">{chips}</div>{out}{bar(r.get("gradient_stops", ""), "all")}{layers}'
            + (f'<img class="dp" src="{dp}">' if dp else "") + "</div>")

place = w.Dropdown(options=["all"] + sorted(lab["place"].unique()), description="place")
state = w.Dropdown(options=["kept", "removed", "all"], description="photos")
reason = w.Dropdown(options=["any"] + sorted(r for r in lab["reason"].unique() if r), description="reason")
n = w.IntSlider(24, 6, 120, 6, description="show")
seed = w.IntText(8, description="shuffle")
out = w.Output()

def render(*_):
    d = lab
    if place.value != "all": d = d[d["place"] == place.value]
    if state.value == "kept": d = d[d["keep"].astype(str).isin(["1", "1.0"])]
    if state.value == "removed": d = d[~d["keep"].astype(str).isin(["1", "1.0"])]
    if reason.value != "any": d = d[d["reason"] == reason.value]
    sample = d.sample(min(n.value, len(d)), random_state=seed.value) if len(d) else d
    with out:
        out.clear_output()
        display(HTML(f"<p><b>{len(d)}</b> photos match · showing {len(sample)}</p>" + CSS +
                     '<div class="lib">' + "".join(card(r) for _, r in sample.iterrows()) + "</div>"))

for c in (place, state, reason, n, seed): c.observe(render, "value")
display(w.HBox([place, state, reason]), w.HBox([n, seed]), out); render()

## 5 · The numbers behind the library

In [ ]:
k = lab[lab["keep"].astype(str).isin(["1", "1.0"])]
print(f"{len(lab)} photos, {len(k)} kept\n")
print("Why photos were removed:")
print(lab[lab["reason"] != ""].groupby(["place", "reason"]).size().unstack(fill_value=0), "\n")
print("Kept, by place and time of day:")
print(pd.crosstab(k["place"], k["time"], margins=True))

## 6 · Save the results

The labels and gradients go back to your Drive (or download), so the autoencoder notebook and the app can use them.
Photos are not included: Places365 photos are not republished.

In [ ]:
import shutil
res = Path("/content/chroma_results"); shutil.rmtree(res, ignore_errors=True); res.mkdir()
for f in ["labels_all.csv", "labels_auto.csv", "closeups.csv", "quality.csv", "palette.csv",
          "gradient_maps.csv", "gradient_layers.csv"]:
    if (DATA / f).exists(): shutil.copy(DATA / f, res / f)
shutil.make_archive("/content/chroma_results", "zip", res)
if Path("/content/drive/MyDrive").exists():
    shutil.copy("/content/chroma_results.zip", "/content/drive/MyDrive/chroma_results.zip")
    print("saved to Drive: MyDrive/chroma_results.zip")
else:
    from google.colab import files; files.download("/content/chroma_results.zip")